# Plot Checkpoint Reconstruction - v19

Select a v19 checkpoint and plot masked event-byte reconstruction quality for compact event chunks. v19 reconstructs only masked event tokens, so the heatmap shows target event bytes and absolute reconstruction error for masked events.

In [1]:
from pathlib import Path

MODEL_VERSION = "v19"
WORKSPACE = None
SAMPLE_CACHE_ROOT = Path(r"D:\market-data\prepared\event_sample_cache")
BATCH_SIZE = 32
EVENTS_PER_CHUNK = 128
DEVICE = "cuda"
USE_AMP = True
CHECKPOINT_PATH = ""
RUNTIME_ROOT = Path(r"D:\TradingML\runtimes\masked_event_model\\v19\pretrain")
ARCHITECTURE_OUTPUT_DIR = Path(r"D:\TradingML\runtimes\masked_event_model\\v19\notebook_artifacts\plot_architecture")
PLOT_SAMPLE_INDEX = 0


In [2]:
import json
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import polars as pl
import torch
from IPython.display import Image, Markdown, SVG, display


def resolve_workspace(explicit):
    if explicit:
        path = Path(explicit)
        if (path / "research" / "masked_event_model" / MODEL_VERSION).exists():
            return path
    candidates = [Path.cwd(), *Path.cwd().parents]
    candidates.extend([
        Path(r"D:\TradingCodes\quant-research-workbench"),
        Path(r"D:\TradingML\codes\masked_event_model\\v19"),
        Path(r"\\DESKTOP-SAAI85T\Workstation-D\TradingML\codes\masked_event_model\\v19"),
    ])
    for path in candidates:
        if (path / "research" / "masked_event_model" / MODEL_VERSION).exists():
            return path
    raise FileNotFoundError("Could not find a workspace root containing research/masked_event_model/v19.")

WORKSPACE = resolve_workspace(WORKSPACE)
if str(WORKSPACE) not in sys.path:
    sys.path.insert(0, str(WORKSPACE))

from research.masked_event_model.v19.config import LossConfig, MaskConfig, ModelConfig
from research.masked_event_model.v19.losses import masked_event_bce_loss, pack_bits
from research.masked_event_model.v19.masking import build_event_masks
from research.masked_event_model.v19.model import EVENT_BYTES, HEADER_BYTES, EventTokenMaskedAutoencoder
from research.mlops.event_sample_cache import EventSampleCacheDataConfig, discover_event_sample_shards, iter_event_sample_cache_epoch_batches, resolve_event_sample_cache_root

plt.style.use("seaborn-v0_8-whitegrid")
plt.rcParams.update({"figure.figsize": (18, 8), "axes.titlesize": 14, "axes.labelsize": 11, "legend.fontsize": 10})
device = torch.device(DEVICE if DEVICE == "cpu" or torch.cuda.is_available() else "cpu")
cache_root = resolve_event_sample_cache_root(SAMPLE_CACHE_ROOT)
print("workspace:", WORKSPACE)
print("cache_root:", cache_root)
print("device:", device)


workspace: d:\TradingCodes\quant-research-workbench
cache_root: D:\market-data\prepared\event_sample_cache
device: cuda


In [3]:
def latest_checkpoint(runtime_root):
    candidates = list(runtime_root.glob("*/checkpoints/latest.pt")) + list(runtime_root.glob("*/checkpoints/checkpoint_latest.pt")) + list(runtime_root.glob("*/checkpoints/*.pt"))
    return max(candidates, key=lambda path: path.stat().st_mtime) if candidates else None

explicit_checkpoint = Path(CHECKPOINT_PATH) if CHECKPOINT_PATH else None
checkpoint_path = explicit_checkpoint if explicit_checkpoint else latest_checkpoint(RUNTIME_ROOT)
if checkpoint_path is not None and checkpoint_path.exists():
    checkpoint = torch.load(checkpoint_path, map_location="cpu")
    checkpoint_config = checkpoint.get("config")
    if not checkpoint_config:
        raise KeyError("Checkpoint does not contain a config payload.")
    model_config = ModelConfig(**{k: v for k, v in checkpoint_config.get("model", {}).items() if k in ModelConfig.__dataclass_fields__})
    mask_config = MaskConfig(**{k: v for k, v in checkpoint_config.get("masks", {}).items() if k in MaskConfig.__dataclass_fields__})
    loss_config = LossConfig(**{k: v for k, v in checkpoint_config.get("losses", {}).items() if k in LossConfig.__dataclass_fields__})
    model = EventTokenMaskedAutoencoder(events_per_chunk=EVENTS_PER_CHUNK, config=model_config).to(device)
    model.load_state_dict(checkpoint.get("model", checkpoint.get("model_state_dict", checkpoint)), strict=True)
    loaded_step = checkpoint.get("step")
    print("checkpoint:", checkpoint_path)
    print("step:", loaded_step)
else:
    if explicit_checkpoint is not None:
        print(f"checkpoint not found: {explicit_checkpoint}")
    else:
        print(f"no checkpoints found under {RUNTIME_ROOT}")
    print("creating a randomly initialized v19 model for architecture/summary plotting")
    checkpoint = None
    checkpoint_path = None
    loaded_step = None
    model_config = ModelConfig()
    mask_config = MaskConfig()
    loss_config = LossConfig()
    model = EventTokenMaskedAutoencoder(events_per_chunk=EVENTS_PER_CHUNK, config=model_config).to(device)
model.eval()
print(f"model parameters={sum(p.numel() for p in model.parameters()):,}")


no checkpoints found under D:\TradingML\runtimes\masked_event_model\v19\pretrain
creating a randomly initialized v19 model for architecture/summary plotting
model parameters=1,498,624


## Dummy Shape Check

Run this cell before loading real sample-cache data to verify that the training path and production encoder path have stable output shapes. The production path uses all events and no masks.


In [4]:
# Dummy train-vs-production shape check.
# This cell intentionally uses synthetic uint8 bytes so it can run without a checkpoint or sample-cache shard.
dummy_batch_size = 2
dummy_header = torch.randint(0, 256, (dummy_batch_size, HEADER_BYTES), dtype=torch.uint8, device=device)
dummy_events = torch.randint(0, 256, (dummy_batch_size, EVENTS_PER_CHUNK, EVENT_BYTES), dtype=torch.uint8, device=device)

model.eval()
dummy_masks = build_event_masks(dummy_events, mask_config)
with torch.inference_mode():
    training_output = model(dummy_header, dummy_events, dummy_masks, mask_config=None)
    production_chunk_embedding = model.encode(dummy_header, dummy_events)
    production_event_embeddings = model.encode_events(dummy_header, dummy_events)

shape_report = {
    "input/header_uint8": tuple(dummy_header.shape),
    "input/events_uint8": tuple(dummy_events.shape),
    "training/visible_event_indices": tuple(dummy_masks.visible_event_indices.shape),
    "training/masked_event_indices": tuple(dummy_masks.masked_event_indices.shape),
    "training/event_bit_logits": tuple(training_output.event_bit_logits.shape),
    "training/chunk_embedding": tuple(training_output.chunk_embedding.shape),
    "production/chunk_embedding": tuple(production_chunk_embedding.shape),
    "production/event_embeddings": tuple(production_event_embeddings.shape),
}
for name, shape in shape_report.items():
    print(f"{name:36s} {shape}")

expected_embedding_shape = (dummy_batch_size, model_config.embedding_dim)
assert tuple(training_output.chunk_embedding.shape) == expected_embedding_shape
assert tuple(production_chunk_embedding.shape) == expected_embedding_shape
assert production_event_embeddings.shape[1] == EVENTS_PER_CHUNK
print("shape check passed")


input/header_uint8                   (2, 14)
input/events_uint8                   (2, 128, 16)
training/visible_event_indices       (2, 38)
training/masked_event_indices        (2, 90)
training/event_bit_logits            (2, 90, 16, 8)
training/chunk_embedding             (2, 32)
production/chunk_embedding           (2, 32)
production/event_embeddings          (2, 128, 32)
shape check passed


In [5]:
# Keras-style model summaries and torchview graphs.
# Optional packages for richer output:
#   pip install torchinfo torchview graphviz
# Graph rendering also needs the Graphviz dot executable on PATH.
#
# The displayed graph is the full MAE training/reconstruction path:
# visible events -> encoder -> chunk_embedding -> projected chunk memory + masked position embedding
# -> per-masked-event MLP -> 16x8 event-bit logits. This mirrors the actual model's train-time
# computation while returning a Tensor instead of EventMAEOutput, which torchinfo
# and torchview cannot inspect directly. The encoder-only production graph is
# still written as an optional secondary artifact for embedding export checks.

from importlib import import_module
import traceback

DRAW_ENCODER_EMBEDDING_GRAPH = globals().get("DRAW_ENCODER_EMBEDDING_GRAPH", True)
DRAW_FULL_TRAINING_RECONSTRUCTION_GRAPH = globals().get("DRAW_FULL_TRAINING_RECONSTRUCTION_GRAPH", True)
ARCHITECTURE_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
architecture_device = torch.device("cpu")
summary_header = torch.zeros((1, HEADER_BYTES), dtype=torch.uint8, device=architecture_device)
summary_events = torch.zeros((1, EVENTS_PER_CHUNK, EVENT_BYTES), dtype=torch.uint8, device=architecture_device)
train_module = import_module(f"research.masked_event_model.{MODEL_VERSION}.train")
EncoderSummaryWrapper = train_module.EncoderSummaryWrapper
MaskedTrainingSummaryWrapper = train_module.MaskedTrainingSummaryWrapper

# Build a fresh CPU-only architecture model. The graph describes structure, not
# trained weights, and this avoids torchview mutating the loaded checkpoint model
# or tracing through CUDA tensors.
architecture_model = EventTokenMaskedAutoencoder(events_per_chunk=EVENTS_PER_CHUNK, config=model_config).to(architecture_device).eval()
summary_models = {
    "full_training_reconstruction": MaskedTrainingSummaryWrapper(architecture_model, EVENTS_PER_CHUNK).to(architecture_device).eval(),
    "encoder_embedding": EncoderSummaryWrapper(architecture_model).to(architecture_device).eval(),
}

try:
    from torchinfo import summary

    for summary_name, summary_model in summary_models.items():
        try:
            summary_text = str(summary(
                summary_model,
                input_data=(summary_header, summary_events),
                depth=8,
                col_names=("input_size", "output_size", "num_params", "trainable"),
                verbose=0,
            ))
            summary_path = ARCHITECTURE_OUTPUT_DIR / f"masked_event_model_{MODEL_VERSION}_{summary_name}_summary.txt"
            summary_path.write_text(summary_text + "\n", encoding="utf-8")
            display(Markdown(f"### {summary_name.replace('_', ' ').title()} Summary"))
            print(summary_text)
            print("summary_path:", summary_path)
        except Exception as exc:
            error_path = ARCHITECTURE_OUTPUT_DIR / f"masked_event_model_{MODEL_VERSION}_{summary_name}_summary_error.txt"
            error_path.write_text("".join(traceback.format_exception(type(exc), exc, exc.__traceback__)), encoding="utf-8")
            print(f"torchinfo summary failed for {summary_name}:", repr(exc))
            print("error_path:", error_path)
except Exception as exc:
    print("torchinfo import failed:", repr(exc))

try:
    from torchview import draw_graph

    graph_names = []
    if DRAW_FULL_TRAINING_RECONSTRUCTION_GRAPH:
        graph_names.append("full_training_reconstruction")
    if DRAW_ENCODER_EMBEDDING_GRAPH:
        graph_names.append("encoder_embedding")
    for graph_name in graph_names:
        graph_model = summary_models[graph_name]
        try:
            graph = draw_graph(graph_model, input_data=(summary_header, summary_events), expand_nested=True, save_graph=False)
            if hasattr(graph, "visual_graph"):
                graph.visual_graph.attr(dpi="220")
                png_path = ARCHITECTURE_OUTPUT_DIR / f"masked_event_model_{MODEL_VERSION}_{graph_name}_torchview.png"
                svg_path = ARCHITECTURE_OUTPUT_DIR / f"masked_event_model_{MODEL_VERSION}_{graph_name}_torchview.svg"
                graph.visual_graph.render(filename=str(png_path.with_suffix("")), directory=str(png_path.parent), format="png", cleanup=True)
                graph.visual_graph.render(filename=str(svg_path.with_suffix("")), directory=str(svg_path.parent), format="svg", cleanup=True)
                print(f"{graph_name}_torchview_png:", png_path)
                print(f"{graph_name}_torchview_svg:", svg_path)
        except Exception as exc:
            error_path = ARCHITECTURE_OUTPUT_DIR / f"masked_event_model_{MODEL_VERSION}_{graph_name}_torchview_error.txt"
            error_path.write_text("".join(traceback.format_exception(type(exc), exc, exc.__traceback__)), encoding="utf-8")
            print(f"torchview graph failed for {graph_name}:", repr(exc))
            print("error_path:", error_path)
    full_png = ARCHITECTURE_OUTPUT_DIR / f"masked_event_model_{MODEL_VERSION}_full_training_reconstruction_torchview.png"
    if full_png.exists():
        display(Markdown("### Full Training/Reconstruction Graph"))
        display(Image(filename=str(full_png)))
except Exception as exc:
    print("torchview import failed:", repr(exc))


### Full Training Reconstruction Summary

Layer (type:depth-idx)                        Input Shape               Output Shape              Param #                   Trainable
MaskedTrainingSummaryWrapper                  [1, 14]                   [1, 32]                   --                        True
├─VisibleEventTokenSelector: 1-1              [1, 128, 16]              [1, 38, 16]               --                        --
├─HeaderTokenEncoder: 1-2                     [1, 14]                   [1, 1, 128]               --                        True
│    └─UInt8BytesToSignedBitFeatures: 2-1     [1, 14]                   [1, 112]                  --                        --
│    └─Sequential: 2-2                        [1, 112]                  [1, 128]                  --                        True
│    │    └─Linear: 3-1                       [1, 112]                  [1, 128]                  14,464                    True
│    │    └─GELU: 3-2                         [1, 128]                  [1, 128]                

### Encoder Embedding Summary

Layer (type:depth-idx)                             Input Shape               Output Shape              Param #                   Trainable
EncoderSummaryWrapper                              [1, 14]                   [1, 32]                   --                        True
├─EventChunkEncoder: 1-1                           [1, 14]                   [1, 32]                   --                        True
│    └─VisibleEventTokenSelector: 2-1              [1, 128, 16]              [1, 128, 16]              --                        --
│    └─HeaderTokenEncoder: 2-2                     [1, 14]                   [1, 1, 128]               --                        True
│    │    └─UInt8BytesToSignedBitFeatures: 3-1     [1, 14]                   [1, 112]                  --                        --
│    │    └─Sequential: 3-2                        [1, 112]                  [1, 128]                  --                        True
│    │    │    └─Linear: 4-1                       [1, 112]  

couldn't load font "Linux libertine Not-Rotated 10", falling back to "Sans Not-Rotated 10", expect ugly output.couldn't load font "Linux libertine Not-Rotated 10", falling back to "Sans Not-Rotated 10", expect ugly output.

full_training_reconstruction_torchview_png: D:\TradingML\runtimes\masked_event_model\v19\notebook_artifacts\plot_architecture\masked_event_model_v19_full_training_reconstruction_torchview.png
full_training_reconstruction_torchview_svg: D:\TradingML\runtimes\masked_event_model\v19\notebook_artifacts\plot_architecture\masked_event_model_v19_full_training_reconstruction_torchview.svg


couldn't load font "Linux libertine Not-Rotated 10", falling back to "Sans Not-Rotated 10", expect ugly output.couldn't load font "Linux libertine Not-Rotated 10", falling back to "Sans Not-Rotated 10", expect ugly output.

encoder_embedding_torchview_png: D:\TradingML\runtimes\masked_event_model\v19\notebook_artifacts\plot_architecture\masked_event_model_v19_encoder_embedding_torchview.png
encoder_embedding_torchview_svg: D:\TradingML\runtimes\masked_event_model\v19\notebook_artifacts\plot_architecture\masked_event_model_v19_encoder_embedding_torchview.svg


### Full Training/Reconstruction Graph

In [ ]:
cache_config = EventSampleCacheDataConfig(
    cache_root=SAMPLE_CACHE_ROOT,
    split="train",
    batch_size=BATCH_SIZE,
    events_per_chunk=EVENTS_PER_CHUNK,
    seed=23,
    prefetch_shards=1,
    max_shards=1,
    shuffle_records=False,
    drop_last=False,
)
shards = discover_event_sample_shards(cache_config)
batch = next(iter_event_sample_cache_epoch_batches(cache_config, epoch=1, shards=shards))
print("batch size:", batch["header_uint8"].shape[0], "shards:", len(shards))
print("first shard:", shards[0].path)


In [ ]:
def move_tensor_batch(batch_dict, device):
    return {key: value.to(device, non_blocking=True) if torch.is_tensor(value) else value for key, value in batch_dict.items()}

device_batch = move_tensor_batch(batch, device)
masks = build_event_masks(device_batch["events_uint8"], mask_config)
with torch.inference_mode(), torch.autocast(device_type=device.type, dtype=torch.float16, enabled=USE_AMP and device.type == "cuda"):
    output = model(device_batch["header_uint8"], device_batch["events_uint8"], masks, mask_config=None)
    result = masked_event_bce_loss(output, loss_config, include_diagnostics=True)
print("loss:", float(result.loss.detach().cpu()))
print(json.dumps(result.metrics, indent=2, sort_keys=True))


In [ ]:
sample = int(PLOT_SAMPLE_INDEX)
event_target = batch["events_uint8"][sample].numpy().astype(float)
event_pred = np.full_like(event_target, np.nan, dtype=float)

masked_event_indices = output.masked_event_indices.detach().cpu()
event_probs = output.event_bit_probs.detach().cpu()
event_bytes = pack_bits(event_probs >= 0.5).numpy() if event_probs.numel() else np.array([])
for event_idx, pred_bytes in zip(masked_event_indices[sample].numpy(), event_bytes[sample]):
    event_pred[int(event_idx), :] = pred_bytes.astype(float)

event_error = np.abs(event_pred - event_target)
fig, axes = plt.subplots(3, 1, figsize=(18, 11), gridspec_kw={"height_ratios": [3, 3, 3]}, constrained_layout=True)
origin_ns = int(batch["origin_timestamp_ns"][sample])
fig.suptitle(f"v19 masked event reconstruction | sample={sample} | origin_ns={origin_ns}", fontweight="bold")
im0 = axes[0].imshow(event_target.T, aspect="auto", interpolation="nearest", cmap="viridis")
axes[0].set_title("target event bytes")
axes[0].set_ylabel("byte index")
fig.colorbar(im0, ax=axes[0], orientation="vertical", fraction=0.015)
im1 = axes[1].imshow(event_pred.T, aspect="auto", interpolation="nearest", cmap="viridis")
axes[1].set_title("reconstructed masked event bytes; unmasked events are blank")
axes[1].set_ylabel("byte index")
fig.colorbar(im1, ax=axes[1], orientation="vertical", fraction=0.015)
im2 = axes[2].imshow(event_error.T, aspect="auto", interpolation="nearest", cmap="magma", vmin=0)
axes[2].set_title("absolute reconstruction error on masked event bytes; unmasked events are blank")
axes[2].set_xlabel("event index")
axes[2].set_ylabel("byte index")
fig.colorbar(im2, ax=axes[2], orientation="vertical", fraction=0.015)
plt.show()


In [ ]:
rows = []
for event_idx, pred_bytes in zip(masked_event_indices[sample].numpy(), event_bytes[sample]):
    event_idx = int(event_idx)
    for byte_idx, value in enumerate(pred_bytes):
        target = int(batch["events_uint8"][sample, event_idx, byte_idx])
        rows.append({"event_index": event_idx, "byte_index": byte_idx, "target_uint8": target, "pred_uint8": int(value), "abs_error": abs(int(value) - target)})
pl.DataFrame(rows).sort(["event_index", "byte_index"]).head(300)
